# 03 Data Wrangling: Transit Access

This notebook creates tract-level transit access features for San Diego County.

I am using SanGIS / SANDAG transit stop data. The goal is to count how many transit stops fall inside each census tract, then create simple access features.

The final output will be:

`transit_access_by_tract.csv`

In [29]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [36]:
# setting up project folders
current_dir = Path.cwd()

raw_data_dir = project_dir / 'data' / 'raw'
processed_data_dir = project_dir / 'data' / 'processed'

raw_data_dir.mkdir(parents=True, exist_ok=True)
processed_data_dir.mkdir(parents=True, exist_ok=True)

print('Project folder:', project_dir)
print('Raw data folder:', raw_data_dir)
print('Processed data folder:', processed_data_dir)

Project folder: C:\Users\cococ\Desktop\Data Science Projects\capstone-3
Raw data folder: C:\Users\cococ\Desktop\Data Science Projects\capstone-3\data\raw
Processed data folder: C:\Users\cococ\Desktop\Data Science Projects\capstone-3\data\processed


In [38]:
# file paths
transit_file = raw_data_dir / 'transit_stops_gtfs' / 'Transit_Stops_GTFS.shp'
tracts_file = raw_data_dir / 'census_tracts' / 'tl_2024_06_tract.shp'
acs_file = processed_data_dir / 'acs_tracts_selected_2024.csv'

print('Transit file exists:', transit_file.exists())
print('Tracts file exists:', tracts_file.exists())
print('ACS file exists:', acs_file.exists())

Transit file exists: True
Tracts file exists: True
ACS file exists: True


In [39]:
# loading San Diego transit stops
transit_stops = gpd.read_file(transit_file)

transit_stops.head()

,stop_UID,stop_agenc,stop_id,stop_name,stop_lat,stop_lon,stop_code,location_t,parent_sta,wheelchair,intersecti,stop_place,geometry
0,MTS_11tbro,MTS,11tbro,11th Ave & Broadway,32.716268,-117.154649,NaN,1,NaN,0.0,NaN,NaN,POINT (6283416.644 1841597.236)
1,MTS_12tS,MTS,12tS,12th & Imperial Station,32.706002,-117.153378,NaN,1,NaN,0.0,NaN,NaN,POINT (6283775.344 1837858.738)
2,MTS_imtS,MTS,imtS,12th & Imperial Station Bayside,32.705229,-117.154318,NaN,1,NaN,0.0,NaN,NaN,POINT (6283483.746 1837580.038)
3,MTS_imp12,MTS,imp12,12th & Imperial Transit Center,32.705685,-117.152875,NaN,1,NaN,0.0,NaN,NaN,POINT (6283929.045 1837742.037)
4,MTS_24tS,MTS,24tS,24th Street Station,32.661854,-117.108017,NaN,1,NaN,0.0,NaN,NaN,POINT (6297596.645 1821678.339)


In [40]:
# checking transit stop data
print('Transit stops shape:', transit_stops.shape)
print('Transit stops CRS:', transit_stops.crs)

transit_stops.info()

Transit stops shape: (6177, 13)
Transit stops CRS: EPSG:2230
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 6177 entries, 0 to 6176
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   stop_UID    6177 non-null   str     
 1   stop_agenc  6177 non-null   str     
 2   stop_id     6177 non-null   str     
 3   stop_name   6177 non-null   str     
 4   stop_lat    6177 non-null   float64 
 5   stop_lon    6177 non-null   float64 
 6   stop_code   6062 non-null   str     
 7   location_t  6177 non-null   str     
 8   parent_sta  306 non-null    str     
 9   wheelchair  6177 non-null   float64 
 10  intersecti  5817 non-null   str     
 11  stop_place  1679 non-null   str     
 12  geometry    6177 non-null   geometry
dtypes: float64(3), geometry(1), str(9)
memory usage: 627.5 KB


In [41]:
# checking columns and geometry type
print(transit_stops.columns.tolist())

transit_stops.geometry.geom_type.value_counts()

['stop_UID', 'stop_agenc', 'stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'stop_code', 'location_t', 'parent_sta', 'wheelchair', 'intersecti', 'stop_place', 'geometry']


Point    6177
Name: count, dtype: int64

In [42]:
# loading California census tract boundaries
tracts = gpd.read_file(tracts_file)

tracts.head()

,STATEFP,COUNTYFP,TRACTCE,GEOID,GEOIDFQ,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,06,001,442700,06001442700,1400000US06001442700,4427,Census Tract 4427,G5020,S,1234016,0,+37.5371513,-122.0081095,"POLYGON ((-122.01721 37.53932, -122.01719 37.5..."
1,06,001,442800,06001442800,1400000US06001442800,4428,Census Tract 4428,G5020,S,1278646,0,+37.5293619,-121.9931002,"POLYGON ((-122.0023 37.52984, -122.00224 37.52..."
2,06,037,204920,06037204920,1400000US06037204920,2049.20,Census Tract 2049.20,G5020,S,909972,0,+34.0175004,-118.1974975,"POLYGON ((-118.20284 34.01966, -118.20283 34.0..."
3,06,037,205110,06037205110,1400000US06037205110,2051.10,Census Tract 2051.10,G5020,S,286962,0,+34.0245059,-118.2142985,"POLYGON ((-118.21964 34.02628, -118.21945 34.0..."
4,06,037,205120,06037205120,1400000US06037205120,2051.20,Census Tract 2051.20,G5020,S,1466242,0,+34.0187542,-118.2117951,"POLYGON ((-118.22023 34.02056, -118.22018 34.0..."


In [43]:
# keeping only San Diego County census tracts
sd_tracts = tracts[tracts['COUNTYFP'] == '073'].copy()

print('California tracts:', tracts.shape)
print('San Diego tracts:', sd_tracts.shape)

sd_tracts.head()

California tracts: (9129, 14)
San Diego tracts: (737, 14)


,STATEFP,COUNTYFP,TRACTCE,GEOID,GEOIDFQ,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
817,06,073,008331,06073008331,1400000US06073008331,83.31,Census Tract 83.31,G5020,S,954207,0,+32.9426037,-117.2241058,"POLYGON ((-117.23082 32.94176, -117.23079 32.9..."
818,06,073,008336,06073008336,1400000US06073008336,83.36,Census Tract 83.36,G5020,S,828562,0,+32.9678415,-117.1331584,"POLYGON ((-117.13793 32.96927, -117.13792 32.9..."
819,06,073,008337,06073008337,1400000US06073008337,83.37,Census Tract 83.37,G5020,S,1572597,0,+32.9583797,-117.1358284,"POLYGON ((-117.14678 32.95497, -117.14657 32.9..."
820,06,073,011601,06073011601,1400000US06073011601,116.01,Census Tract 116.01,G5020,S,733715,0,+32.6663314,-117.0964015,"POLYGON ((-117.10356 32.6672, -117.10314 32.66..."
821,06,073,011602,06073011602,1400000US06073011602,116.02,Census Tract 116.02,G5020,S,1005857,13822,+32.6599903,-117.0941732,"POLYGON ((-117.10154 32.66202, -117.10133 32.6..."


In [44]:
# checking tract data before spatial join
print('Tracts CRS:', sd_tracts.crs)

sd_tracts[['GEOID', 'COUNTYFP', 'TRACTCE', 'NAMELSAD']].head()

Tracts CRS: EPSG:4269


,GEOID,COUNTYFP,TRACTCE,NAMELSAD
817,06073008331,073,008331,Census Tract 83.31
818,06073008336,073,008336,Census Tract 83.36
819,06073008337,073,008337,Census Tract 83.37
820,06073011601,073,011601,Census Tract 116.01
821,06073011602,073,011602,Census Tract 116.02


In [45]:
# loading ACS data so we can use population later if needed
acs = pd.read_csv(acs_file)

acs['geo_id'] = acs['geo_id'].astype(str)

print('ACS shape:', acs.shape)
acs.head()

ACS shape: (737, 50)


,geo_id,tract_name,total_population,population_under_5,population_under_5_rate,population_under_18,population_under_18_rate,population_65_plus,median_age,hispanic_latino_population,hispanic_latino_rate,total_households,households_with_children,households_with_seniors,avg_household_size,avg_family_size,school_enrolled_population,elementary_school_enrollment,high_school_enrollment,bachelors_or_higher,median_household_income,mean_household_income,poverty_rate,family_poverty_rate,employed_population,unemployed_population,unemployment_rate,drove_alone_count,drove_alone_rate,public_transit_commute_count,public_transit_commute_rate,work_from_home_count,work_from_home_rate,total_housing_units,occupied_housing_units,vacant_housing_units,vacancy_rate,owner_occupied_units,renter_occupied_units,renter_rate,vehicle_households,no_vehicle_households,no_vehicle_rate,one_vehicle_households,two_vehicle_households,three_plus_vehicle_households,median_gross_rent,rent_burden_30_34_count,rent_burden_35_plus_count,tract_type_flag
0,1400000US06073000100,Census Tract 1; San Diego County; California,2948,175,5.9,657,22.3,815,51.1,281,9.5,1178,318,543,2.50,2.89,716,224,229,1758,231667.0,331240.0,2.2,2.1,1435,0,0.0,996,69.8,25,1.8,269,18.9,1293,1178,115,8.9,1067,111,9.4,1178,41,3.5,260,592,285,NaN,17,20,residential_or_mixed
1,1400000US06073000201,Census Tract 2.01; San Diego County; California,2270,80,3.5,440,19.4,711,51.2,90,4.0,1180,208,577,1.92,3.03,459,212,111,1392,124722.0,224327.0,6.7,6.5,1063,45,2.3,538,51.0,7,0.7,361,34.3,1261,1180,81,6.4,534,646,54.7,1180,148,12.5,557,353,122,2407.0,59,233,residential_or_mixed
2,1400000US06073000202,Census Tract 2.02; San Diego County; California,3755,107,2.8,296,7.9,753,45.8,562,15.0,2240,185,557,1.64,2.44,343,99,39,2077,120091.0,144236.0,2.9,0.9,2418,174,5.0,1424,58.0,117,4.8,816,33.2,2476,2240,236,9.5,1003,1237,55.2,2240,99,4.4,1210,703,228,1992.0,134,375,residential_or_mixed
3,1400000US06073000301,Census Tract 3.01; San Diego County; California,2311,93,4.0,187,8.1,357,37.3,516,22.3,1302,131,250,1.77,2.55,244,32,27,1248,87813.0,121130.0,15.4,18.1,1435,88,4.1,839,56.9,87,5.9,457,31.0,1475,1302,173,11.7,345,957,73.5,1302,81,6.2,683,436,102,1945.0,69,316,residential_or_mixed
4,1400000US06073000302,Census Tract 3.02; San Diego County; California,2873,0,0.0,73,2.5,731,45.1,450,15.7,1818,63,535,1.48,1.99,131,38,0,1644,89573.0,125762.0,9.4,0.0,1810,63,2.3,895,49.1,77,4.2,506,27.8,2197,1818,379,17.3,501,1317,72.4,1818,122,6.7,1290,342,64,2412.0,80,664,residential_or_mixed


In [46]:
print(transit_stops.crs)
print(sd_tracts.crs)

EPSG:2230
EPSG:4269


Similar to the crime data, the transit data is in a different format so we need to standardize.

In [47]:
# matching tract CRS to transit stops CRS before spatial join
sd_tracts = sd_tracts.to_crs(transit_stops.crs)

print('Transit CRS:', transit_stops.crs)
print('Tracts CRS:', sd_tracts.crs)

Transit CRS: EPSG:2230
Tracts CRS: EPSG:2230


In [48]:
# calculating tract area in square miles
sd_tracts['tract_area_sq_mile'] = sd_tracts.geometry.area / 27878400

sd_tracts[['GEOID', 'NAMELSAD', 'tract_area_sq_mile']].head()

,GEOID,NAMELSAD,tract_area_sq_mile
817,06073008331,Census Tract 83.31,0.368403
818,06073008336,Census Tract 83.36,0.319892
819,06073008337,Census Tract 83.37,0.607150
820,06073011601,Census Tract 116.01,0.283300
821,06073011602,Census Tract 116.02,0.393717


## Coordinate system and standardization

Before joining the transit stops to census tracts, I checked the coordinate systems for both files, which were different. In order for me to do a spatial join, I had to match them up.

The transit stops were in `EPSG:2230`, and the census tracts were in `EPSG:4269`. Since `EPSG:2230` is the local system for San Diego, I made that the default.

Then I calculated each tract’s area since `EPSG:2230` is in feet, but I want my scores to be per square mile for `transit_stops_per_sq_mile`.

conversion: 27,878,400 square feet tp 1 square mile.

This gives a more fair comparison between tracts. A large tract might have more transit stops just because it covers more land. Using stops per square mile helps adjust for tract size.


In [49]:
# joining each transit stop to the census tract it falls inside
transit_with_tract = gpd.sjoin(
    transit_stops,
    sd_tracts[['GEOID', 'NAMELSAD', 'tract_area_sq_mile', 'geometry']],
    how='left',
    predicate='within')

transit_with_tract.head()

,stop_UID,stop_agenc,stop_id,stop_name,stop_lat,stop_lon,stop_code,location_t,parent_sta,wheelchair,intersecti,stop_place,geometry,index_right,GEOID,NAMELSAD,tract_area_sq_mile
0,MTS_11tbro,MTS,11tbro,11th Ave & Broadway,32.716268,-117.154649,NaN,1,NaN,0.0,NaN,NaN,POINT (6283416.644 1841597.236),2306,06073005202,Census Tract 52.02,0.090329
1,MTS_12tS,MTS,12tS,12th & Imperial Station,32.706002,-117.153378,NaN,1,NaN,0.0,NaN,NaN,POINT (6283775.344 1837858.738),2112,06073005103,Census Tract 51.03,1.018244
2,MTS_imtS,MTS,imtS,12th & Imperial Station Bayside,32.705229,-117.154318,NaN,1,NaN,0.0,NaN,NaN,POINT (6283483.746 1837580.038),2112,06073005103,Census Tract 51.03,1.018244
3,MTS_imp12,MTS,imp12,12th & Imperial Transit Center,32.705685,-117.152875,NaN,1,NaN,0.0,NaN,NaN,POINT (6283929.045 1837742.037),2112,06073005103,Census Tract 51.03,1.018244
4,MTS_24tS,MTS,24tS,24th Street Station,32.661854,-117.108017,NaN,1,NaN,0.0,NaN,NaN,POINT (6297596.645 1821678.339),5398,06073021900,Census Tract 219,3.668896


In [51]:
# checking how many transit stops matched to a tract
matched_stops = transit_with_tract['GEOID'].notna().sum()
unmatched_stops = transit_with_tract['GEOID'].isna().sum()
total_stops = len(transit_with_tract)

print('Total transit stops:', total_stops)
print('Matched to tract:', matched_stops)
print('Match rate:', round(matched_stops / total_stops * 100, 2), '%')

Total transit stops: 6177
Matched to tract: 6177
Match rate: 100.0 %


In [52]:
# counting transit stops by census tract
transit_counts = (
    transit_with_tract
    .groupby('GEOID')
    .size()
    .reset_index(name='transit_stop_count'))

transit_counts.head()


,GEOID,transit_stop_count
0,06073000100,6
1,06073000201,7
2,06073000202,11
3,06073000301,7
4,06073000302,10


In [53]:
# joining transit counts back to all San Diego tracts
transit_by_tract = sd_tracts.merge(
    transit_counts,
    on='GEOID',
    how='left')

# tracts with no stops should be counted as 0
transit_by_tract['transit_stop_count'] = transit_by_tract['transit_stop_count'].fillna(0).astype(int)

transit_by_tract[['GEOID', 'NAMELSAD', 'tract_area_sq_mile', 'transit_stop_count']].head()

,GEOID,NAMELSAD,tract_area_sq_mile,transit_stop_count
0,06073008331,Census Tract 83.31,0.368403,0
1,06073008336,Census Tract 83.36,0.319892,0
2,06073008337,Census Tract 83.37,0.607150,0
3,06073011601,Census Tract 116.01,0.283300,10
4,06073011602,Census Tract 116.02,0.393717,13


In [55]:
# calculating transit stop density by tract
transit_by_tract['transit_stops_per_sq_mile'] = (
    transit_by_tract['transit_stop_count'] / transit_by_tract['tract_area_sq_mile'])

transit_by_tract[
    ['GEOID', 'NAMELSAD', 'tract_area_sq_mile', 'transit_stop_count', 'transit_stops_per_sq_mile']].head()

,GEOID,NAMELSAD,tract_area_sq_mile,transit_stop_count,transit_stops_per_sq_mile
0,06073008331,Census Tract 83.31,0.368403,0,0.000000
1,06073008336,Census Tract 83.36,0.319892,0,0.000000
2,06073008337,Census Tract 83.37,0.607150,0,0.000000
3,06073011601,Census Tract 116.01,0.283300,10,35.298246
4,06073011602,Census Tract 116.02,0.393717,13,33.018628


In [57]:
# checking the tract-level transit features
transit_by_tract[
    ['GEOID', 'NAMELSAD', 'tract_area_sq_mile', 'transit_stop_count', 
     'transit_stops_per_sq_mile', 'has_transit_access']].head()

,GEOID,NAMELSAD,tract_area_sq_mile,transit_stop_count,transit_stops_per_sq_mile,has_transit_access
0,06073008331,Census Tract 83.31,0.368403,0,0.000000,0
1,06073008336,Census Tract 83.36,0.319892,0,0.000000,0
2,06073008337,Census Tract 83.37,0.607150,0,0.000000,0
3,06073011601,Census Tract 116.01,0.283300,10,35.298246,1
4,06073011602,Census Tract 116.02,0.393717,13,33.018628,1


In [58]:
# checking missing values in the tract-level transit data
transit_missing = (
    transit_by_tract[
        ['GEOID', 'NAMELSAD', 'tract_area_sq_mile', 'transit_stop_count',
         'transit_stops_per_sq_mile', 'has_transit_access']].isna().sum().sort_values(ascending=False).to_frame('missing_count'))

transit_missing['missing_percent'] = (
    transit_missing['missing_count'] / len(transit_by_tract) * 100).round(2)

transit_missing

,missing_count,missing_percent
GEOID,0,0.0
NAMELSAD,0,0.0
tract_area_sq_mile,0,0.0
transit_stop_count,0,0.0
transit_stops_per_sq_mile,0,0.0
has_transit_access,0,0.0


In [59]:
# checking the distribution of the transit features
transit_by_tract[
    ['tract_area_sq_mile', 'transit_stop_count', 
     'transit_stops_per_sq_mile', 'has_transit_access']
].describe()

,tract_area_sq_mile,transit_stop_count,transit_stops_per_sq_mile,has_transit_access
count,737.000000,737.000000,737.000000,737.000000
mean,6.140450,8.381275,14.156623,0.880597
std,38.845248,8.840976,17.751393,0.324482
min,0.085692,0.000000,0.000000,0.000000
25%,0.417017,3.000000,1.780954,1.000000
50%,0.727669,7.000000,9.372666,1.000000
75%,1.457847,11.000000,20.830933,1.000000
max,631.505778,90.000000,243.554768,1.000000


In [60]:
# looking at the largest tracts to see if any are unusual
transit_by_tract[
    ['GEOID', 'NAMELSAD', 'tract_area_sq_mile', 
     'transit_stop_count', 'transit_stops_per_sq_mile', 'has_transit_access']].sort_values('tract_area_sq_mile', ascending=False).head(10)

,GEOID,NAMELSAD,tract_area_sq_mile,transit_stop_count,transit_stops_per_sq_mile,has_transit_access
189,06073021001,Census Tract 210.01,631.505778,12,0.019002,1
370,06073020903,Census Tract 209.03,499.617447,12,0.024018,1
145,06073021002,Census Tract 210.02,433.760829,7,0.016138,1
280,06073990100,Census Tract 9901,254.635664,0,0.000000,0
183,06073021101,Census Tract 211.01,234.708210,37,0.157643,1
195,06073021102,Census Tract 211.02,217.517820,16,0.073557,1
399,06073018700,Census Tract 187,215.867505,81,0.375230,1
469,06073020902,Census Tract 209.02,166.315608,4,0.024051,1
365,06073021302,Census Tract 213.02,147.575231,7,0.047433,1
211,06073019108,Census Tract 191.08,76.460322,19,0.248495,1


In [61]:
# flagging unusually large tracts for later review
large_tract_threshold = transit_by_tract['tract_area_sq_mile'].quantile(0.95)

transit_by_tract['large_tract_flag'] = np.where(
    transit_by_tract['tract_area_sq_mile'] >= large_tract_threshold,
    1,0)

print('Large tract threshold:', round(large_tract_threshold, 2), 'sq miles')

transit_by_tract[
    ['GEOID', 'NAMELSAD', 'tract_area_sq_mile', 'large_tract_flag']].sort_values('tract_area_sq_mile', ascending=False).head(10)

Large tract threshold: 12.56 sq miles


,GEOID,NAMELSAD,tract_area_sq_mile,large_tract_flag
189,06073021001,Census Tract 210.01,631.505778,1
370,06073020903,Census Tract 209.03,499.617447,1
145,06073021002,Census Tract 210.02,433.760829,1
280,06073990100,Census Tract 9901,254.635664,1
183,06073021101,Census Tract 211.01,234.708210,1
195,06073021102,Census Tract 211.02,217.517820,1
399,06073018700,Census Tract 187,215.867505,1
469,06073020902,Census Tract 209.02,166.315608,1
365,06073021302,Census Tract 213.02,147.575231,1
211,06073019108,Census Tract 191.08,76.460322,1


In [62]:
# checking how many tracts have at least one transit stop
transit_by_tract['has_transit_access'].value_counts()

has_transit_access
1    649
0     88
Name: count, dtype: int64

In [63]:
# checking transit access as percentages
transit_by_tract['has_transit_access'].value_counts(normalize=True).mul(100).round(2)

has_transit_access
1    88.06
0    11.94
Name: proportion, dtype: float64

In [66]:
# keeping the final tract-level transit columns
transit_output = transit_by_tract[
    [
        'GEOID',
        'NAMELSAD',
        'tract_area_sq_mile',
        'transit_stop_count',
        'transit_stops_per_sq_mile',
        'has_transit_access',
        'large_tract_flag']].copy()

transit_output.head()

,GEOID,NAMELSAD,tract_area_sq_mile,transit_stop_count,transit_stops_per_sq_mile,has_transit_access,large_tract_flag
0,06073008331,Census Tract 83.31,0.368403,0,0.000000,0,0
1,06073008336,Census Tract 83.36,0.319892,0,0.000000,0,0
2,06073008337,Census Tract 83.37,0.607150,0,0.000000,0,0
3,06073011601,Census Tract 116.01,0.283300,10,35.298246,1,0
4,06073011602,Census Tract 116.02,0.393717,13,33.018628,1,0


In [68]:
# final check before saving
print('Final transit output shape:', transit_output.shape)

Final transit output shape: (737, 7)


In [69]:
# saving final tract-level transit access file
output_file = processed_data_dir / 'transit_access_by_tract.csv'

transit_output.to_csv(output_file, index=False)

print('Saved to:', output_file)

Saved to: C:\Users\cococ\Desktop\Data Science Projects\capstone-3\data\processed\transit_access_by_tract.csv


In [70]:
# reloading saved file to make sure it exported correctly
saved_transit = pd.read_csv(output_file)

print('Saved transit shape:', saved_transit.shape)
saved_transit.head()

Saved transit shape: (737, 7)


,GEOID,NAMELSAD,tract_area_sq_mile,transit_stop_count,transit_stops_per_sq_mile,has_transit_access,large_tract_flag
0,6073008331,Census Tract 83.31,0.368403,0,0.000000,0,0
1,6073008336,Census Tract 83.36,0.319892,0,0.000000,0,0
2,6073008337,Census Tract 83.37,0.607150,0,0.000000,0,0
3,6073011601,Census Tract 116.01,0.283300,10,35.298246,1,0
4,6073011602,Census Tract 116.02,0.393717,13,33.018628,1,0


## Transit wrangling summary

I used SanGIS / SANDAG transit stop data and matched each stop to a census tract. All of the stops matched to a tract, so the spatial join looked good.

After that, I grouped the data by tract and counted how many transit stops were in each one. I also calculated stops per square mile so the comparison is a little more fair between small and large tracts.

I kept the features simple for now since this is still the wrangling stage. I added a flag for whether a tract has any transit stops, plus a flag for unusually large tracts that I may want to review later in EDA.

The final output was saved as:

`transit_access_by_tract.csv`

I'll reference this file later with the ACS and crime/safety files to build the full neighborhood opportunity dataset.
